# 多租户 LoRA Serving：从逐请求加载到版本化 Adapter Pool

**面试问题：一个共享基座怎样批量服务多个 adapter，并处理版本、缓存、租户隔离和热更新？**

## 回答主线

先把真实请求和资源合同摆出来，用最简单的方案建立成本或正确性基线，再手写核心控制逻辑并展示完整事件、指标和失败修正。断言只在最后保护少量关键不变量，前面的可见输入、过程和结果才是学习主体。

## 真实案例

一家 SaaS 客服平台共享同一个 7B 基座，但服装、银行和物流三个租户分别有 LoRA adapter。五个在线请求会乱序到达，GPU adapter pool 只能容纳两个版本。案例用小矩阵做真实 forward，对比逐请求加载与驻留池，展示批内分组、LRU、refcount、不可变热更新，以及错误租户或错误 base revision 的拒绝路径。

### 输入预览：三个租户、五个请求和两槽 GPU

In [1]:
import numpy as np  # 导入矩阵运算以展示 base 与 LoRA delta 的真实 forward。

base_revision = "base-7b-r4"  # 固定共享基座版本，避免 adapter 挂到错误权重上。
base_weight = np.array([[0.8, -0.2, 0.3], [0.1, 0.6, -0.4]], dtype=float)  # 构造把三个输入特征映射到两个 logits 的基座权重。
adapter_specs = {  # 定义租户、版本、低秩矩阵和基座依赖。
    "fashion:v3": {"tenant": "fashion", "base": base_revision, "a": np.array([[0.2, 0.0, -0.1]]), "b": np.array([[0.5], [-0.2]]), "scale": 0.8},  # 服装租户强调退换货特征。
    "bank:v2": {"tenant": "bank", "base": base_revision, "a": np.array([[0.0, 0.3, 0.1]]), "b": np.array([[-0.3], [0.7]]), "scale": 1.0},  # 银行租户强调风险与账户特征。
    "logistics:v1": {"tenant": "logistics", "base": base_revision, "a": np.array([[-0.1, 0.1, 0.4]]), "b": np.array([[0.6], [0.1]]), "scale": 0.6},  # 物流租户强调时效特征。
}  # 完成不可变 adapter registry。
requests = [("r1", "fashion", "fashion:v3", np.array([1.0, 0.0, 0.4])), ("r2", "bank", "bank:v2", np.array([0.1, 1.0, 0.2])), ("r3", "fashion", "fashion:v3", np.array([0.6, 0.0, 0.8])), ("r4", "logistics", "logistics:v1", np.array([0.0, 0.2, 1.0])), ("r5", "bank", "bank:v2", np.array([0.2, 0.9, 0.5]))]  # 构造五条跨租户乱序请求并让被驱逐版本再次加载。
print("请求批次：")  # 输出输入预览标题。
for request_id, tenant, adapter_id, features in requests:  # 逐条展示路由需要的身份和特征。
    print(f"{request_id} tenant={tenant:<9} adapter={adapter_id:<12} features={features}")  # 显示同租户请求可以共享已加载 adapter。

请求批次：
r1 tenant=fashion   adapter=fashion:v3   features=[1.  0.  0.4]
r2 tenant=bank      adapter=bank:v2      features=[0.1 1.  0.2]
r3 tenant=fashion   adapter=fashion:v3   features=[0.6 0.  0.8]
r4 tenant=logistics adapter=logistics:v1 features=[0.  0.2 1. ]
r5 tenant=bank      adapter=bank:v2      features=[0.2 0.9 0.5]


## Baseline 基线：每个请求都从 CPU 重新加载 Adapter

In [2]:
def lora_forward(features, base, spec):  # 实现 base linear 加低秩 LoRA delta 的完整前向。
    base_output = base @ features  # 计算共享基座的两个输出 logits。
    low_rank = spec["a"] @ features  # 把输入投影到一维低秩空间。
    delta = spec["scale"] * (spec["b"] @ low_rank)  # 把低秩表示映射回输出并应用缩放。
    return base_output + delta  # 返回租户 adapter 调整后的最终 logits。

baseline_load_ms = 0.0  # 初始化逐请求加载产生的累计延迟。
baseline_outputs = {}  # 保存每个请求的 reference 输出。
for request_id, tenant, adapter_id, features in requests:  # 逐请求模拟最简单的加载与推理流程。
    baseline_load_ms += 12.0  # 假设每次从 CPU 搬运一个小 adapter 需要十二毫秒。
    baseline_outputs[request_id] = lora_forward(features, base_weight, adapter_specs[adapter_id])  # 计算正确但加载低效的 reference 输出。
print(f"逐请求加载次数={len(requests)}，累计 adapter 搬运延迟={baseline_load_ms:.1f} ms")  # 展示基线没有复用同租户 adapter。
print("服装请求 r1 输出：", np.round(baseline_outputs["r1"], 4))  # 展示真实 base+delta 数值而不只检查 shape。

逐请求加载次数=5，累计 adapter 搬运延迟=60.0 ms
服装请求 r1 输出： [ 0.984  -0.0856]


### 核心实现：LRU、refcount 与租户/基座门禁

In [3]:
class AdapterPool:  # 定义容量受限且可审计的 adapter 驻留池。
    def __init__(self, capacity, registry, expected_base):  # 初始化容量、可信 registry 和基座版本。
        self.capacity = capacity  # 保存 GPU 可同时驻留的 adapter 数量。
        self.registry = registry  # 保存不可变 adapter 元数据和权重来源。
        self.expected_base = expected_base  # 保存当前服务实例加载的基座版本。
        self.loaded = {}  # 记录 adapter 最近访问时钟和引用计数。
        self.clock = 0  # 使用单调时钟实现确定性 LRU。
        self.events = []  # 保存 load、hit、evict 与 reject 事件供调试。

    def acquire(self, tenant, adapter_id):  # 为一个请求获取经过授权和兼容检查的 adapter。
        self.clock += 1  # 每次访问推进逻辑时钟。
        spec = self.registry.get(adapter_id)  # 从可信 registry 查找精确不可变版本。
        if spec is None or spec["tenant"] != tenant:  # 拒绝不存在或跨租户使用的 adapter。
            self.events.append((self.clock, "reject_tenant", adapter_id))  # 记录身份拒绝事件。
            raise PermissionError("adapter 不属于当前租户")  # 返回明确授权错误而不是静默回退。
        if spec["base"] != self.expected_base:  # 检查 adapter 是否与当前基座 revision 兼容。
            self.events.append((self.clock, "reject_base", adapter_id))  # 记录版本不兼容事件。
            raise ValueError("adapter 与 base revision 不兼容")  # 阻止 shape 相同但语义错误的加载。
        if adapter_id not in self.loaded:  # 未驻留时需要寻找空槽或安全驱逐。
            candidates = [(state["last"], key) for key, state in self.loaded.items() if state["refs"] == 0]  # 只允许驱逐没有 in-flight 引用的 adapter。
            if len(self.loaded) >= self.capacity and not candidates:  # 所有槽位都被运行中请求占用时必须排队。
                self.events.append((self.clock, "queue", adapter_id))  # 记录池满排队事件。
                raise RuntimeError("adapter pool 暂无安全槽位")  # 避免驱逐仍在 GPU kernel 使用的权重。
            if len(self.loaded) >= self.capacity:  # 有可驱逐候选时选择最久未使用版本。
                _, victim = min(candidates)  # 根据逻辑时钟确定 LRU victim。
                self.loaded.pop(victim)  # 从驻留表移除 victim。
                self.events.append((self.clock, "evict", victim))  # 保存驱逐事件用于抖动分析。
            self.loaded[adapter_id] = {"last": self.clock, "refs": 0}  # 为新 adapter 建立驻留状态。
            self.events.append((self.clock, "load", adapter_id))  # 记录一次真实搬运。
        else:  # adapter 已驻留时直接命中缓存。
            self.events.append((self.clock, "hit", adapter_id))  # 记录复用事件。
        self.loaded[adapter_id]["refs"] += 1  # 增加引用计数保护当前请求。
        self.loaded[adapter_id]["last"] = self.clock  # 更新最近使用时间。
        return spec  # 返回经过授权和版本检查的 adapter。

    def release(self, adapter_id):  # 在请求 kernel 完成后释放引用。
        self.loaded[adapter_id]["refs"] -= 1  # 减少引用计数使 adapter 重新可被驱逐。

pool = AdapterPool(2, adapter_specs, base_revision)  # 创建只有两个 GPU 槽位的教学池。
pooled_outputs = {}  # 保存驻留池路径的推理输出。
for request_id, tenant, adapter_id, features in requests:  # 按到达顺序执行五个请求。
    spec = pool.acquire(tenant, adapter_id)  # 获取授权且兼容的 adapter。
    pooled_outputs[request_id] = lora_forward(features, base_weight, spec)  # 使用共享 base 和当前 delta 前向。
    pool.release(adapter_id)  # kernel 完成后释放 adapter 引用。
print("Adapter Pool 事件：")  # 输出完整 load/hit/evict 时间线。
for event in pool.events:  # 逐条渲染控制面事件。
    print(event)  # 展示第二个 fashion 请求命中以及第三租户触发安全驱逐。

Adapter Pool 事件：
(1, 'load', 'fashion:v3')
(2, 'load', 'bank:v2')
(3, 'hit', 'fashion:v3')
(4, 'evict', 'bank:v2')
(4, 'load', 'logistics:v1')
(5, 'evict', 'fashion:v3')
(5, 'load', 'bank:v2')


## 结果解读：批内复用、输出等价和搬运成本

In [4]:
load_events = [event for event in pool.events if event[1] == "load"]  # 统计真正发生权重搬运的事件。
hit_events = [event for event in pool.events if event[1] == "hit"]  # 统计 adapter cache 命中事件。
pooled_load_ms = len(load_events) * 12.0  # 使用相同搬运成本估算驻留池延迟。
max_output_error = max(float(np.max(np.abs(pooled_outputs[key] - baseline_outputs[key]))) for key in baseline_outputs)  # 验证调度优化没有改变模型数学输出。
print("请求  adapter       最终logits")  # 输出逐请求结果表表头。
for request_id, _, adapter_id, _ in requests:  # 逐请求展示 adapter 身份和最终 logits。
    print(f"{request_id:<5} {adapter_id:<13} {np.round(pooled_outputs[request_id], 4)}")  # 让学习者看到不同租户 delta 的真实效果。
print(f"load={len(load_events)}，hit={len(hit_events)}，搬运延迟 {baseline_load_ms:.1f}->{pooled_load_ms:.1f} ms，输出最大误差={max_output_error:.1e}")  # 汇总性能收益与正确性 oracle。

请求  adapter       最终logits
r1    fashion:v3    [ 0.984  -0.0856]
r2    bank:v2       [-0.156  0.754]
r3    fashion:v3    [ 0.736  -0.2664]
r4    logistics:v1  [ 0.4112 -0.2548]
r5    bank:v2       [0.034 0.584]
load=4，hit=1，搬运延迟 60.0->48.0 ms，输出最大误差=0.0e+00


## 失败案例：跨租户请求和错误 Base 热更新

In [5]:
failure_events = []  # 收集两种高风险失败的可读结果。
try:  # 尝试让银行租户调用服装 adapter。
    pool.acquire("bank", "fashion:v3")  # 发起跨租户越权获取。
except PermissionError as error:  # 捕获预期授权拒绝。
    failure_events.append(("跨租户", str(error)))  # 保存拒绝原因供客户端和审计使用。
bad_spec = dict(adapter_specs["bank:v2"])  # 复制银行 adapter 元数据构造错误热更新。
bad_spec["base"] = "base-7b-r5"  # 把新 adapter 绑定到尚未部署的基座版本。
adapter_specs["bank:v3"] = bad_spec  # 向 registry 注册不兼容的新版本用于演示门禁。
try:  # 尝试在旧 base 服务实例加载新 adapter。
    pool.acquire("bank", "bank:v3")  # 发起 shape 可能相同但语义不兼容的加载。
except ValueError as error:  # 捕获预期版本拒绝。
    failure_events.append(("错误基座", str(error)))  # 保存版本拒绝原因。
print("失败与修正结果：")  # 输出安全失败表标题。
for failure in failure_events:  # 逐项展示系统没有静默回退到错误权重。
    print(failure)  # 呈现明确的越权和版本错误。

失败与修正结果：
('跨租户', 'adapter 不属于当前租户')
('错误基座', 'adapter 与 base revision 不兼容')


### 生产边界

In [6]:
metrics = {"adapter_loads": len(load_events), "adapter_hits": len(hit_events), "evictions": sum(event[1] == "evict" for event in pool.events), "resident": sorted(pool.loaded), "base_revision": base_revision}  # 汇总低基数服务指标和版本状态。
print("服务指标：", metrics)  # 展示生产监控需要关注的缓存抖动与命中。
print("生产替换点：真实系统还需要 pinned memory、异步 DMA、按 adapter 分组 kernel、连续批处理、不可变制品摘要和跨副本目录。")  # 明确小矩阵池与 GPU Serving 的差距。

服务指标： {'adapter_loads': 4, 'adapter_hits': 1, 'evictions': 2, 'resident': ['bank:v2', 'logistics:v1'], 'base_revision': 'base-7b-r4'}
生产替换点：真实系统还需要 pinned memory、异步 DMA、按 adapter 分组 kernel、连续批处理、不可变制品摘要和跨副本目录。


## 回归测试：只保护等价、复用和隔离

In [7]:
assert max_output_error < 1e-12  # 验证 adapter pool 调度不改变 base+LoRA 数学输出。
assert len(hit_events) == 1  # 验证第二个服装请求真实复用了驻留 adapter。
assert len(pool.loaded) <= pool.capacity  # 验证任何时刻驻留数量都没有突破 GPU 槽位。
assert len(failure_events) == 2  # 验证跨租户和错误基座两类风险都被明确拒绝。
assert all(state["refs"] == 0 for state in pool.loaded.values())  # 验证请求结束后没有泄漏 in-flight 引用。
print("回归测试通过：输出等价、缓存复用、容量、租户隔离和引用释放均成立。")  # 用少量断言总结核心服务合同。

回归测试通过：输出等价、缓存复用、容量、租户隔离和引用释放均成立。
